In [ ]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings("ignore")

# 导入必要的基础类
from sklearn.base import BaseEstimator, RegressorMixin, clone
from sklearn.model_selection import KFold
from sklearn.multioutput import MultiOutputRegressor

# 导入各种回归器
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.ensemble import ExtraTreesRegressor
from sklearn import metrics

# 导入其他回归模型
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from sklearn.neighbors import KNeighborsRegressor
import torch
from tabpfn import TabPFNRegressor  # 添加TabPFN导入
from pytorch_tabnet.tab_model import TabNetRegressor
# 检查是否有 GPU 可用
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"使用设备: {device}")
# 创建TabNet的包装器，使其符合scikit-learn的接口
# 定义改进后的多输出堆叠回归器

# TabNet包装器类，使其符合scikit-learn的接口
class TabNetRegressorWrapper(BaseEstimator, RegressorMixin):
    def __init__(self, n_d=22, n_a=128, n_steps=3, gamma=2.0, lambda_sparse=0.0005467072969055289, 
                 optimizer_fn=torch.optim.Adam, optimizer_params=dict(lr=0.029384293039604333), 
                 scheduler_params={"step_size":50, "gamma":0.8}, scheduler_fn=torch.optim.lr_scheduler.StepLR, 
                 mask_type='entmax', batch_size=227, virtual_batch_size=96, max_epochs=200, 
                 patience=20, device_name='auto', verbose=0, seed=42):
        self.n_d = n_d
        self.n_a = n_a
        self.n_steps = n_steps
        self.gamma = gamma
        self.lambda_sparse = lambda_sparse
        self.optimizer_fn = optimizer_fn
        self.optimizer_params = optimizer_params
        self.scheduler_params = scheduler_params
        self.scheduler_fn = scheduler_fn
        self.mask_type = mask_type
        self.batch_size = batch_size
        self.virtual_batch_size = virtual_batch_size
        self.max_epochs = max_epochs
        self.patience = patience
        self.device_name = device_name
        self.verbose = verbose
        self.seed = seed
        self.model = None
        
    def fit(self, X, y):
        if len(y.shape) == 1:
            y = y.reshape(-1, 1)
            
        # 创建模型
        self.model = TabNetRegressor(
            n_d=self.n_d,
            n_a=self.n_a,
            n_steps=self.n_steps,
            gamma=self.gamma,
            lambda_sparse=self.lambda_sparse,
            optimizer_fn=self.optimizer_fn,
            optimizer_params=self.optimizer_params,
            scheduler_params=self.scheduler_params,
            scheduler_fn=self.scheduler_fn,
            mask_type=self.mask_type,
            device_name=self.device_name,
            verbose=self.verbose,
            seed=self.seed
        )
        
        # 抑制输出
        import contextlib
        import io
        with contextlib.redirect_stdout(io.StringIO()):
            # 训练模型
            self.model.fit(
                X_train=X,
                y_train=y,
                max_epochs=self.max_epochs,
                patience=self.patience,
                batch_size=self.batch_size,
                virtual_batch_size=self.virtual_batch_size,
                num_workers=0
            )
        return self
    
    def predict(self, X):
        if self.model is None:
            raise ValueError("Model not fitted yet.")
        preds = self.model.predict(X)
        if preds.shape[1] == 1:
            return preds.ravel()
        return preds


from sklearn.compose import TransformedTargetRegressor
from sklearn.pipeline import make_pipeline

def make_fold_estimator(estimator):
    return TransformedTargetRegressor(
        regressor=make_pipeline(StandardScaler(), clone(estimator)),
        transformer=StandardScaler()
    )


class MultiOutputStackingRegressor(BaseEstimator, RegressorMixin):
    def __init__(self, estimators, final_estimator=None, cv=5, n_jobs=-1):
        self.estimators = estimators
        self.final_estimator = final_estimator
        self.cv = cv
        self.n_jobs = n_jobs

    def fit(self, X, y):
        self.n_outputs_ = y.shape[1]
        self.estimators_ = []
        self.final_estimator_ = clone(self.final_estimator)
        X = np.asarray(X)
        
        # 设置交叉验证折叠
        kfold = KFold(n_splits=self.cv, shuffle=True, random_state=42)
        
        # 用于存储所有目标变量和所有模型的元特征
        meta_features = np.zeros((X.shape[0], 0))
        
        # 对每个基础学习器
        for name, estimator in self.estimators:
            # 为每个输出变量创建一个模型
            wrapped_estimators = []
            estimator_meta_features = np.zeros((X.shape[0], 0))
            
            # 对每个输出变量单独创建和训练模型
            for j in range(self.n_outputs_):
                # 用于存储当前估计器的当前输出变量的折叠预测
                est_meta_features = np.zeros((X.shape[0], 1))
                
                # 训练当前特定输出变量的模型并生成元特征
                for train_idx, val_idx in kfold.split(X):
                    X_train_fold, X_val_fold = X[train_idx], X[val_idx]
                    y_train_fold = y[train_idx, j].reshape(-1, 1)
                    
                    # 为当前输出变量克隆一个估计器
                    fold_est = make_fold_estimator(estimator)
                    # 训练模型
                    fold_est.fit(X_train_fold, y_train_fold.ravel())
                    
                    # 生成预测并保存为元特征 - 修复形状不匹配问题
                    preds = fold_est.predict(X_val_fold)
                    # 确保预测是一维数组
                    if isinstance(preds, np.ndarray) and len(preds.shape) > 1:
                        preds = preds.ravel()
                    est_meta_features[val_idx, 0] = preds
                
                # 将当前输出变量的元特征添加到当前估计器的总元特征中
                estimator_meta_features = np.hstack((estimator_meta_features, est_meta_features))
                
                # 在全部数据上训练一个针对当前输出变量的最终模型
                final_est = make_fold_estimator(estimator)
                final_est.fit(X, y[:, j])
                wrapped_estimators.append((f"{name}_{j}", final_est))
            
            # 将当前估计器对应的元特征添加到总的元特征中
            meta_features = np.hstack((meta_features, estimator_meta_features))
            
            # 保存当前基础模型对应的所有输出变量模型
            self.estimators_.append((name, wrapped_estimators))
        
        # 训练元模型
        if self.final_estimator_ is not None:
            self.final_estimator_.fit(meta_features, y)
            
        return self

    def predict(self, X):
        # 生成元特征
        meta_features = self._get_meta_features(X)
        
        # 使用元模型进行预测
        if self.final_estimator_ is not None:
            return self.final_estimator_.predict(meta_features)
        else:
            # 如果没有元模型，则手动组合各个基础模型的预测
            predictions = np.zeros((X.shape[0], self.n_outputs_))
            
            for _, wrapped_estimators in self.estimators_:
                for i, (_, model) in enumerate(wrapped_estimators):
                    predictions[:, i % self.n_outputs_] += model.predict(X)
            
            return predictions / len(self.estimators)

    def _get_meta_features(self, X):
        """生成元特征用于元模型预测"""
        meta_features = np.zeros((X.shape[0], 0))
        
        # 获取每个基础模型的每个输出变量的预测作为元特征
        for _, wrapped_estimators in self.estimators_:
            estimator_meta_features = np.zeros((X.shape[0], 0))
            
            for _, model in wrapped_estimators:
                # 获取预测并确保其是2D数组
                preds = model.predict(X)
                if len(preds.shape) == 1:
                    preds = preds.reshape(-1, 1)
                estimator_meta_features = np.hstack((estimator_meta_features, preds))
            
            meta_features = np.hstack((meta_features, estimator_meta_features))
                
        return meta_features

# 导入原始数据
df_original = pd.read_excel('../../data/quyi_data_x_names.xlsx')
print(f"原始数据集大小: {df_original.shape}")

# 提取目标变量
target_vars = ['x25', 'x26', 'x27', 'x28', 'x29']

# 从原始数据中分离特征和目标
X_original = df_original.drop(target_vars, axis=1)
y_original = df_original[target_vars]

# 将原始数据分为训练集和测试集
# Paper protocol: preserve furnace order; first 900 records train, next 500 test.
if len(X_original) != 1400 or len(y_original) != 1400:
    raise ValueError('Expected 1,400 ordered furnace records (900 train + 500 test).')
X_train, X_test = X_original.iloc[:900].copy(), X_original.iloc[900:].copy()
y_train, y_test = y_original.iloc[:900].copy(), y_original.iloc[900:].copy()

print(f"训练集大小: {X_train.shape}")
print(f"测试集大小: {X_test.shape}")

# 创建标准化器
# Fold-local preprocessing is handled by the base-estimator pipelines.
X_train_array = X_train.to_numpy(dtype=float)
y_train_array = y_train.to_numpy(dtype=float)
X_test_array = X_test.to_numpy(dtype=float)
y_test_array = y_test.to_numpy(dtype=float)

base_learners = [
    ("RF", RandomForestRegressor(
        n_estimators=200,  # 减少估计器数量以提高速度
        max_depth=20,
        min_samples_split=2,
        min_samples_leaf=1,
        max_features="sqrt",  # 使用sqrt特征以加速
        n_jobs=-1,
        random_state=42
    )),("ET",ExtraTreesRegressor(
    n_estimators=500,    
    max_depth=30,        
    min_samples_split=2, 
    min_samples_leaf=1,  
    max_features=None,   
    bootstrap=False,     
    n_jobs=-1,           
    random_state=42
)),
("KNN",KNeighborsRegressor(
    n_neighbors=2,  # 邻居数量，可以进行调整
    weights='distance',  # 权重类型：'uniform'或'distance'
    algorithm='auto',  # 算法：'auto', 'ball_tree', 'kd_tree', 'brute'
    p=1,  # 距离度量，2表示欧氏距离
    n_jobs=-1  # 使用所有可用的CPU核心
)),("LGBM", LGBMRegressor(
        boosting_type= 'gbdt',
        objective= 'regression',
        metric= 'rmse',
        num_leaves= 64,
        max_depth= 14,
        learning_rate= 0.10279735193954564,
        feature_fraction= 0.792740064845325,
        subsample= 0.7533616827994455,
        min_child_samples= 17,
        subsample_freq= 5,
        verbose= -1,
        n_jobs= -1,
        n_estimators= 335,
        random_state= 42
    )),
 ("TabNet", TabNetRegressorWrapper(
        n_d=22,                      # 决策层维度
        n_a=128,                     # 注意力层维度
        n_steps=3,                   # 步数
        gamma=2.0,                   # 稀疏度
        lambda_sparse=0.0005467072969055289,  # 额外的稀疏度
        optimizer_fn=torch.optim.Adam,
        optimizer_params=dict(lr=0.029384293039604333),  # 学习率
        scheduler_params={"step_size":50, "gamma":0.8},
        scheduler_fn=torch.optim.lr_scheduler.StepLR,
        mask_type='entmax',
        device_name=device,
        batch_size=227,
        virtual_batch_size=96,
        max_epochs=100,  # 减少训练轮数以加快训练速度
        patience=20,
        verbose=0,
        seed=42
    )),
    # ("TabPFN", TabPFNRegressor(device=device,)),
    
]

# 定义二级学习器
meta_model = Ridge(alpha=1.0, random_state=42)

# 创建改进的多输出堆叠回归器，使用5折交叉验证
stacking_regressor = MultiOutputStackingRegressor(
    estimators=base_learners, 
    final_estimator=meta_model,
    cv=2
)

# 进行交叉验证评估
print("\n开始进行交叉验证评估...")
k_folds = 2
kf = KFold(n_splits=k_folds, shuffle=True, random_state=42)

cv_scores = {var: [] for var in target_vars}
cv_rmse = {var: [] for var in target_vars}
cv_mae = {var: [] for var in target_vars}

for train_idx, val_idx in kf.split(X_train_array):
    # 分割训练集和验证集
    X_train_fold, X_val_fold = X_train_array[train_idx], X_train_array[val_idx]
    y_train_fold, y_val_fold = y_train_array[train_idx], y_train_array[val_idx]
    
    # 训练堆叠模型
    stacking_regressor.fit(X_train_fold, y_train_fold)
    
    # 在验证集上预测
    y_val_pred = stacking_regressor.predict(X_val_fold)
    
    # 计算每个目标变量的指标
    for i, var in enumerate(target_vars):
        r2 = metrics.r2_score(y_val_fold[:, i], y_val_pred[:, i])
        rmse = np.sqrt(metrics.mean_squared_error(y_val_fold[:, i], y_val_pred[:, i]))
        mae = metrics.mean_absolute_error(y_val_fold[:, i], y_val_pred[:, i])
        
        cv_scores[var].append(r2)
        cv_rmse[var].append(rmse)
        cv_mae[var].append(mae)

# 打印交叉验证结果
print(f"\n{k_folds}折交叉验证结果:")
mean_cv_r2 = 0
mean_cv_rmse = 0
mean_cv_mae = 0

for var in target_vars:
    mean_r2 = np.mean(cv_scores[var])
    mean_rmse = np.mean(cv_rmse[var])
    mean_mae = np.mean(cv_mae[var])
    
    mean_cv_r2 += mean_r2
    mean_cv_rmse += mean_rmse
    mean_cv_mae += mean_mae
    
    print(f"\n{var} 交叉验证指标:")
    print(f"平均 R²: {mean_r2:.4f} ± {np.std(cv_scores[var]):.4f}")
    print(f"平均 RMSE: {mean_rmse:.4f} ± {np.std(cv_rmse[var]):.4f}")
    print(f"平均 MAE: {mean_mae:.4f} ± {np.std(cv_mae[var]):.4f}")

# 计算所有目标变量的平均指标
mean_cv_r2 /= len(target_vars)
mean_cv_rmse /= len(target_vars)
mean_cv_mae /= len(target_vars)

print("\n所有变量平均交叉验证指标:")
print(f"平均 R²: {mean_cv_r2:.4f}")
print(f"平均 RMSE: {mean_cv_rmse:.4f}")
print(f"平均 MAE: {mean_cv_mae:.4f}")

# 在完整训练集上训练最终堆叠模型
print("\n开始训练最终堆叠模型...")
stacking_regressor.fit(X_train_array, y_train_array)
print("堆叠模型训练完成!")

# 使用训练好的模型在测试集上预测
y_pred_test_array = stacking_regressor.predict(X_test_array)

# 将预测结果反标准化回原始空间
y_pred_test = y_pred_test_array

# 真实值
y_test_true = y_test.values

# 为每个输出目标计算评估指标
print("\n堆叠模型测试集评估指标:")
overall_r2 = 0
overall_rmse = 0
overall_mae = 0

for idx, name in enumerate(target_vars):
    # 测试集指标
    mse_test = metrics.mean_squared_error(y_test_true[:, idx], y_pred_test[:, idx])
    rmse_test = np.sqrt(mse_test)
    mae_test = metrics.mean_absolute_error(y_test_true[:, idx], y_pred_test[:, idx])
    r2_test = metrics.r2_score(y_test_true[:, idx], y_pred_test[:, idx])
    
    overall_r2 += r2_test
    overall_rmse += rmse_test
    overall_mae += mae_test
    
    print(f"\n{name} 指标:")
    print(f"MSE={mse_test:.4f}, RMSE={rmse_test:.4f}, MAE={mae_test:.4f}, R²={r2_test:.4f}")
    print('-----------------------------------')

# 计算所有目标的平均性能指标
overall_r2 /= len(target_vars)
overall_rmse /= len(target_vars)
overall_mae /= len(target_vars)

print("\n总体平均指标:")
print(f"平均 RMSE: {overall_rmse:.4f}")
print(f"平均 MAE: {overall_mae:.4f}")
print(f"平均 R²: {overall_r2:.4f}")

# 与原始模型比较
print("\n与原始无交叉验证模型的性能比较:")
print("改进后的模型(使用交叉验证)平均 R²: {:.4f}".format(overall_r2))